# Análisis de resultados — suite de evaluación

Lee el CSV más reciente generado por `src.evals.run` (modo full, 30 partidos) y produce:

1. Métricas agregadas sobre los 30 partidos.
2. Breakdown por tier (top / mid / bottom).
3. Mejores y peores casos por cada métrica.
4. Distribución de formaciones propuestas vs reales.
5. Errores si los hay.

Este notebook va a ser la base del análisis de resultados de Miguel y Daniel para la documentación.

In [1]:
import glob
import json
from pathlib import Path

import pandas as pd

for candidate in [Path('evals/results'), Path('../evals/results')]:
    if candidate.exists():
        RESULTS = candidate.resolve()
        break

full_runs = sorted(glob.glob(str(RESULTS / 'results_full_*.csv')))
if not full_runs:
    raise FileNotFoundError('No hay results_full_*.csv. Corre src.evals.run primero.')
latest = full_runs[-1]
df = pd.read_csv(latest)
print(f'Archivo: {Path(latest).name}')
print(f'Partidos evaluados: {len(df)}')
print(f'Columnas: {len(df.columns)}')

Archivo: results_full_20260528_103048.csv
Partidos evaluados: 30
Columnas: 21


## 1. Métricas agregadas

In [2]:
METRIC_COLS = ['lineup_overlap', 'formation_match', 'tactical_coherence', 'specificity', 'faithfulness']
df[METRIC_COLS].describe().round(2)

,lineup_overlap,formation_match,tactical_coherence,specificity,faithfulness
count,30.00,30.00,30.00,30.0,30.00
mean,0.58,0.37,3.83,4.1,3.17
std,0.18,0.49,0.59,0.4,0.83
min,0.27,0.00,3.00,3.0,2.00
25%,0.39,0.00,3.25,4.0,3.00
50%,0.64,0.00,4.00,4.0,3.00
75%,0.70,1.00,4.00,4.0,4.00
max,0.91,1.00,5.00,5.0,5.00


## 2. Breakdown por tier

¿El sistema rinde igual para equipos top, mid y bottom?

In [3]:
df.groupby('tier')[METRIC_COLS].mean().round(2).reindex(['top', 'mid', 'bottom'])

,lineup_overlap,formation_match,tactical_coherence,specificity,faithfulness
tier,,,,,
top,0.44,0.5,3.8,3.9,3.3
mid,0.65,0.5,3.9,4.1,2.9
bottom,0.64,0.1,3.8,4.3,3.3


In [4]:
df.groupby('tier')[METRIC_COLS].std().round(2).reindex(['top', 'mid', 'bottom'])

,lineup_overlap,formation_match,tactical_coherence,specificity,faithfulness
tier,,,,,
top,0.14,0.53,0.63,0.32,0.82
mid,0.13,0.53,0.74,0.32,0.88
bottom,0.19,0.32,0.42,0.48,0.82


## 3. Mejores y peores casos

Para entender qué falla y qué funciona, vemos los 3 partidos con mejor y peor puntaje en cada dimensión.

In [5]:
cols = ['tier', 'our_team', 'opponent', 'faithfulness', 'faithfulness_why']
print('=== Mejor faithfulness ===')
df.nlargest(3, 'faithfulness')[cols].to_string(index=False)

=== Mejor faithfulness ===


'tier    our_team   opponent  faithfulness                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             faithfulness_why\n top Real Madrid Celta Vigo             5                                                                                                                                                                                Todos los datos citados en el plan (PPDA 14.51, xG en contra 56.3, 48 goles encajados, 263 deep completions) están presentes y son exactos respecto al scouting validado. No se inventa ningún número ni se atribuyen características 

In [6]:
print('=== Peor faithfulness ===')
df.nsmallest(3, 'faithfulness')[cols].to_string(index=False)

=== Peor faithfulness ===


"tier       our_team        opponent  faithfulness                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     faithfulness_why\n top     Celta Vigo Atletico Madrid             2                                                                              El plan cita correctamente el PPDA (11.68), los xG del partido Celta-Atlético (0.29 vs 2.79) y las deep completions (351), todos presentes en el scouting. Sin embargo, introduce datos no respaldados por el sc

In [7]:
cols2 = ['tier', 'our_team', 'opponent', 'lineup_overlap', 'formation_proposed', 'formation_actual']
print('=== Mejor lineup_overlap ===')
df.nlargest(3, 'lineup_overlap')[cols2].to_string(index=False)

=== Mejor lineup_overlap ===


'  tier our_team    opponent  lineup_overlap formation_proposed formation_actual\nbottom  Osasuna      Alaves           0.909              4-5-1            4-5-1\nbottom Mallorca    Valencia           0.909              4-4-2            4-5-1\n   mid   Getafe Real Oviedo           0.818              5-3-2            5-3-2'

In [8]:
print('=== Peor lineup_overlap ===')
df.nsmallest(3, 'lineup_overlap')[cols2].to_string(index=False)

=== Peor lineup_overlap ===


'  tier        our_team opponent  lineup_overlap formation_proposed formation_actual\n   top Atletico Madrid   Getafe           0.273              4-3-3            4-5-1\nbottom         Sevilla    Elche           0.273              4-3-3            3-4-3\n   top      Celta Vigo  Levante           0.364              3-4-3            3-4-3'

## 4. Formaciones — propuestas vs reales

In [9]:
print('Formaciones más propuestas por el Coach:')
print(df['formation_proposed'].value_counts().to_string())
print()
print('Formaciones reales (detectadas desde ESPN):')
print(df['formation_actual'].value_counts().to_string())

Formaciones más propuestas por el Coach:
formation_proposed
4-3-3      10
4-4-2       7
3-4-3       5
4-5-1       5
4-2-3-1     2
5-3-2       1

Formaciones reales (detectadas desde ESPN):
formation_actual
4-5-1    15
3-4-3     5
4-4-2     5
4-3-3     2
5-3-2     2
3-5-2     1


In [10]:
pd.crosstab(df['formation_actual'], df['formation_proposed'], margins=True, margins_name='Total')

formation_proposed,3-4-3,4-2-3-1,4-3-3,4-4-2,4-5-1,5-3-2,Total
formation_actual,,,,,,,
3-4-3,4,0,1,0,0,0,5
3-5-2,0,0,1,0,0,0,1
4-3-3,0,0,1,0,1,0,2
4-4-2,1,1,1,2,0,0,5
4-5-1,0,1,6,5,3,0,15
5-3-2,0,0,0,0,1,1,2
Total,5,2,10,7,5,1,30


## 5. Errores

In [11]:
errs = df[df['error'].fillna('') != '']
if errs.empty:
    print('Sin errores en los 30 partidos.')
else:
    print(f'{len(errs)} partidos con error:')
    print(errs[['tier', 'our_team', 'opponent', 'error']].to_string(index=False))

Sin errores en los 30 partidos.


## Notas para Miguel y Daniel

- Las métricas estructuradas (`lineup_overlap`, `formation_match`) son objetivas y reproducibles.
- Las métricas del juez (`tactical_coherence`, `specificity`, `faithfulness`) son subjetivas — un modelo (Sonnet 4.6) puntuando otro (Haiku 4.5). Hay que documentar esta limitación.
- `faithfulness` está medida contra el scouting visible al juez, NO contra el ground truth completo (decisión metodológica del v1).
- `lineup_overlap` está sub-medido por desajustes de nombre entre Understat (lo que ve el Coach) y ESPN (el ground truth). Una mejora futura sería fuzzy match con rapidfuzz.